# Pointwise Mutual Information: What Retrieval Adds to Generation, in Bits

This narrative notebook imports the canonical, tested reference implementation
`pmi_retrieval_value.py` and walks the topic section by section. The `.py` **owns every number**;
here we just call it so the claims render as executed output.

The evaluation layer scored retrieval by *whether the right document came back*. This topic scores it by
a different question — *how many bits does a retrieved document add to the generator's answer?* Generation
begins with an answer prior $p(a\mid q)$, uncertain with entropy $H(A\mid Q)$; a retrieved document $d$
sharpens it to a posterior $p(a\mid q,d)$, and the log-ratio is the **pointwise mutual information**
$\mathrm{pmi}(a;d\mid q)=\log_2 p(a\mid q,d)/p(a\mid q)$.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks" / "pmi-retrieval-value"))
import numpy as np
import pmi_retrieval_value as pmi

c = pmi._corpus()
print("K (answers/documents) =", c["K"], " N_QUERIES =", c["n_queries"], " DIM =", pmi.PMI_DIM)
print("sector_of_passage     =", [int(s) for s in c["sector_of_passage"]])
print("tau =", pmi.TAU, " tau_doc =", pmi.TAU_DOC, " kappa_query =", pmi.PMI_KAPPA_QUERY)

## Movement 0 — the corpus: reuse the finance geometry, ask a sector-ambiguous question

We **reuse** the dense-retrieval finance vMF geometry — four sectors, two companies each, one filing per
company — so the answer prototypes *are* the company document directions. The queries are drawn around the
**sector mean** (concentration $\kappa_q$), so a query pins the sector but not the company: the gold
answer is the in-sector company the query is nearest, and the *other* in-sector company is a plausible
distractor.

In [ ]:
# The geometry is the dense-retrieval topic's, asserted byte-for-byte; same-sector companies are near,
# cross-sector near-orthogonal (the vMF sizing rule).
pmi.test_corpus_reuses_dpr_geometry()
wq = pmi.pick_worked_query(c)
a_star = int(c["truth"][wq]); d_id = pmi._distractor_id(c, wq)
print(f"worked query {wq}: sector {int(c['q_sector'][wq])}, gold company a* = {a_star}, "
      f"same-sector distractor = {d_id}")

## Movement 1 — the answer model: prior, posterior, and the log-ratio between them

$p(d\mid q)=\mathrm{softmax}(\langle q,P_d\rangle/\tau_{\mathrm{doc}})$ is the dual-encoder MIPS
distribution; $p(a\mid q,d)=\mathrm{softmax}((\langle q,\mathrm{proto}_a\rangle+\langle d,
\mathrm{proto}_a\rangle)/\tau)$ adds the document's evidence to the query's; and the prior is the **RAG
marginal** $p(a\mid q)=\sum_d p(d\mid q)\,p(a\mid q,d)$.

In [ ]:
qd = pmi.query_distributions(c, wq)
print("prior p(a|q)            :", np.round(qd["prior"], 3))
print("H(prior)                :", round(pmi.entropy(qd["prior"]), 3), "bits")
post_rel = pmi.answer_posterior(qd["q"], c["P"][a_star], c["protos"])
post_dis = pmi.answer_posterior(qd["q"], c["P"][d_id], c["protos"])
print("posterior | relevant doc:", np.round(post_rel, 3), " H =", round(pmi.entropy(post_rel), 3))
print("posterior | distractor  :", np.round(post_dis, 3), " H =", round(pmi.entropy(post_dis), 3))
print("pmi(a*; relevant   | q) :", round(pmi.pmi_pointwise(qd["prior"], post_rel, a_star), 3), "bits")
print("pmi(a*; distractor | q) :", round(pmi.pmi_pointwise(qd["prior"], post_dis, a_star), 3), "bits")

## Movement 2 — information gain and the three-way-verified identity

The per-document gain is $\mathrm{KL}(p(\cdot\mid q,d)\,\|\,p(\cdot\mid q))\ge 0$, and the
conditional mutual information $I(A;D\mid Q)=H(A\mid Q)-H(A\mid Q,D)=\mathbb{E}[\mathrm{KL}]$ — the bits
retrieval adds. The notebook verifies all three computational routes agree to numerical zero.

In [ ]:
pmi.test_three_way_mi_agreement()
b = pmi.cond_mi_breakdown(c)
disagree = max(abs(b["expected_kl"] - b["entropy_reduction"]),
               abs(b["entropy_reduction"] - b["joint_sum"]))
print(f"H(A|Q) = {b['H_A_given_Q']:.4f} bits   H(A|Q,D) = {b['H_A_given_QD']:.4f} bits")
print(f"I(A;D|Q): expected_kl={b['expected_kl']:.6f}  entropy_reduction={b['entropy_reduction']:.6f}  "
      f"joint_sum={b['joint_sum']:.6f}")
print(f"max pairwise disagreement = {disagree:.2e}")

## Movement 3 — a relevant filing adds bits; a distractor costs them

A relevant filing puts **positive** pmi on the true answer; a same-sector distractor puts **negative** pmi
there — it drags the posterior toward the wrong company, costing bits — even though its
$\mathrm{KL}(\text{post}\,\|\,\text{prior})>0$ (it still moves belief, just wrongly).

In [ ]:
pmi.test_pmi_sign_separation()
t = pmi.pmi_at_truth_table(c)
print(f"mean pmi at a*, relevant   doc = {t['mean_rel']:+.3f} bits")
print(f"mean pmi at a*, distractor doc = {t['mean_distr']:+.3f} bits")
print(f"fraction of distractors with pmi(a*) < 0 = {t['frac_distr_negative']:.1%}")
print(f"mean distractor KL(post||prior) = {float(np.mean(t['kl_distr'])):.3f} bits (> 0: belief moved)")

## Movement 4 — diminishing returns: a redundant second document adds almost nothing

A second *identical* filing barely moves belief; a *different* filing moves it a lot — the chain rule of
mutual information made visible, and the reason context selection (a forward topic) is a coverage problem.

In [ ]:
pmi.test_saturation_diminishing_returns()
s = pmi.saturation_table(c)
print(f"standalone (gold, 1st doc)        = {s['standalone']:.3f} bits")
print(f"redundant  (a 2nd identical doc)  = {s['redundant']:.3f} bits  <- saturation")
print(f"novel      (a 2nd different doc)  = {s['novel']:.3f} bits")

## Movement 5 — the encoder maximized a lower bound on these bits; and bits $\neq$ recall

InfoNCE is a lower bound on $I(Q;D)$ **ceilinged at $\log_2(N{+}1)$**: even a perfect encoder certifies at
most $\log_2(N{+}1)$ bits with $N$ negatives, so the bound *saturates* as we add candidates. And on this
easy-retrieval corpus recall is pinned at $1.0$ for every query, yet the bits retrieval adds vary
widely — *presence is not contribution*.

In [ ]:
pmi.test_infonce_bound_saturates(); pmi.test_bits_vs_recall_differ()
inf = pmi.infonce_bound_curve(c)
for r in inf["rows"]:
    print(f"  m={r['m']:2d}  L={r['L']:.3f}  ceiling=log2(m)={r['ceiling_bits']:.3f}  "
          f"bound={r['bound_bits']:.3f}  (bound <= ceiling)")
bvr = pmi.bits_vs_recall_table(c)
print(f"recall@3 in [{bvr['recall_min']}, {bvr['recall_max']}] (saturated), MAP={bvr['map']:.3f}")
print(f"bits added in [{bvr['bits_min']:.3f}, {bvr['bits_max']:.3f}], std={bvr['bits_std']:.3f} (vary)")

## Every claim, verified

The full harness runs each pedagogical claim as an assertion and prints the constants the interactive lab
mirrors to the decimal.

In [ ]:
pmi._run_all()